In [0]:
%sql

--deleting null values in customerids
DELETE FROM retail_catalog.bronze.customers_raw
WHERE CustomerID IS NULL;

In [0]:
%sql
--deleting null values of customer id in silver layer
DELETE FROM retail_catalog.silver.dim_customer
WHERE CustomerID IS NULL;

In [0]:
%sql

--store id mismatch
WITH mismatches AS (
  -- Find bronze store IDs not in silver
  SELECT COUNT(*) AS missing_in_target
  FROM retail_catalog.bronze.stores_raw b
  LEFT JOIN retail_catalog.silver.dim_store s
    ON b.StoreID = s.StoreID
  WHERE s.StoreID IS NULL
),
orphans AS (
  -- Find silver store IDs not in bronze
  SELECT COUNT(*) AS missing_in_source
  FROM retail_catalog.silver.dim_store s
  LEFT JOIN retail_catalog.bronze.stores_raw b
    ON s.StoreID = b.StoreID
  WHERE b.StoreID IS NULL
)
SELECT 
  m.missing_in_target,
  o.missing_in_source,
  CASE 
    WHEN m.missing_in_target = 0 AND o.missing_in_source = 0 THEN 'PASS: All store IDs match'
    ELSE assert_true(FALSE, CONCAT('FAIL: ', CAST(m.missing_in_target AS STRING), ' store IDs missing in target, ', CAST(o.missing_in_source AS STRING), ' store IDs missing in source'))
  END AS validation_status
FROM mismatches m, orphans o;

In [0]:
%sql

--details of datatypes of customer table
DESC retail_catalog.silver.dim_customer;

In [0]:
%sql
DESCRIBE retail_catalog.silver.fact_sales;

In [0]:
%sql
SELECT b.CustomerID
FROM retail_catalog.bronze.customers_raw b

LEFT JOIN retail_catalog.silver.dim_customer s
ON b.CustomerID = s.CustomerID
AND s.IsActive = 1

WHERE s.CustomerID IS NULL;

In [0]:
%sql
SELECT b.TransactionID
FROM retail_catalog.bronze.sales_raw b
LEFT JOIN retail_catalog.silver.fact_sales s
ON b.TransactionID = s.TransactionID
WHERE s.TransactionID IS NULL;

In [0]:
%sql
SELECT
    CustomerID,
    COUNT(*) AS active_count
FROM retail_catalog.silver.dim_customer
WHERE IsActive = 1
GROUP BY CustomerID
HAVING COUNT(*) > 1;

In [0]:
%sql
WITH duplicates AS (
  SELECT
    ProductID,
    COUNT(*) AS duplicate_count
  FROM retail_catalog.silver.dim_product
  GROUP BY ProductID
  HAVING COUNT(*) > 1
),
duplicate_summary AS (
  SELECT 
    COUNT(*) AS total_duplicates,
    SUM(duplicate_count) AS total_duplicate_records
  FROM duplicates
)
SELECT 
  total_duplicates,
  total_duplicate_records,
  CASE 
    WHEN total_duplicates = 0 THEN 'PASS: No duplicate ProductIDs found'
    ELSE assert_true(FALSE, CONCAT('FAIL: Found ', CAST(total_duplicates AS STRING), ' duplicate ProductID(s) with ', CAST(total_duplicate_records AS STRING), ' total duplicate records'))
  END AS validation_status
FROM duplicate_summary;

In [0]:
%sql
WITH negative_amounts AS (
  SELECT COUNT(*) AS negative_count
  FROM retail_catalog.silver.fact_sales
  WHERE Amount < 0
)
SELECT 
  negative_count,
  CASE 
    WHEN negative_count = 0 THEN 'PASS: No negative amounts found'
    ELSE assert_true(FALSE, CONCAT('FAIL: Found ', CAST(negative_count AS STRING), ' records with negative amounts'))
  END AS validation_status
FROM negative_amounts;

In [0]:
%sql
WITH duplicates AS (
  SELECT
    CustomerID,
    COUNT(*) AS duplicate_count
  FROM retail_catalog.silver.dim_customer
  WHERE IsActive = 1
  GROUP BY CustomerID
  HAVING COUNT(*) > 1
),
duplicate_summary AS (
  SELECT 
    COUNT(*) AS total_duplicates,
    SUM(duplicate_count) AS total_duplicate_records
  FROM duplicates
)
SELECT 
  total_duplicates,
  total_duplicate_records,
  CASE 
    WHEN total_duplicates = 0 THEN 'PASS: No duplicate active CustomerIDs found'
    ELSE assert_true(FALSE, CONCAT('FAIL: Found ', CAST(total_duplicates AS STRING), ' duplicate active CustomerID(s) with ', CAST(total_duplicate_records AS STRING), ' total duplicate records'))
  END AS validation_status
FROM duplicate_summary;

In [0]:
%sql
WITH invalid_enddate AS (
  SELECT COUNT(*) AS invalid_count
  FROM retail_catalog.silver.dim_customer
  WHERE IsActive = 1
  AND EndDate != DATE('9999-12-31')
)
SELECT 
  invalid_count,
  CASE 
    WHEN invalid_count = 0 THEN 'PASS: All active customers have correct EndDate'
    ELSE assert_true(FALSE, CONCAT('FAIL: Found ', CAST(invalid_count AS STRING), ' active customer(s) with incorrect EndDate (expected 9999-12-31)'))
  END AS validation_status
FROM invalid_enddate;

In [0]:
%sql
--source to target validation customers
WITH counts AS (
  SELECT 
    (SELECT COUNT(*) FROM retail_catalog.bronze.customers_raw) AS source_count,
    (SELECT COUNT(*) FROM retail_catalog.silver.dim_customer WHERE IsActive = 1) AS target_count
)
SELECT 
  source_count,
  target_count,
  CASE 
    WHEN source_count = target_count THEN 'PASS: Counts match'
    ELSE assert_true(FALSE, CONCAT('FAIL: Source count (', CAST(source_count AS STRING), ') does not match target count (', CAST(target_count AS STRING), ')'))
  END AS validation_status
FROM counts;

In [0]:
%sql

--source to target validation products
WITH counts AS (
  SELECT 
    (SELECT COUNT(*) FROM retail_catalog.bronze.products_raw) AS source_count,
    (SELECT COUNT(*) FROM retail_catalog.silver.dim_product) AS target_count
)
SELECT 
  source_count,
  target_count,
  CASE 
    WHEN source_count = target_count THEN 'PASS: Counts match'
    ELSE assert_true(FALSE, CONCAT('FAIL: Source count (', CAST(source_count AS STRING), ') does not match target count (', CAST(target_count AS STRING), ')'))
  END AS validation_status
FROM counts;

In [0]:
%sql

--source to target validation sales
WITH counts AS (
  SELECT 
    (SELECT COUNT(*) FROM retail_catalog.bronze.sales_raw) AS source_count,
    (SELECT COUNT(*) FROM retail_catalog.silver.fact_sales) AS target_count
)
SELECT 
  source_count,
  target_count,
  CASE 
    WHEN source_count = target_count THEN 'PASS: Counts match'
    ELSE assert_true(FALSE, CONCAT('FAIL: Source count (', CAST(source_count AS STRING), ') does not match target count (', CAST(target_count AS STRING), ')'))
  END AS validation_status
FROM counts;

In [0]:
%sql

--source to target validation customer id mismatch
WITH mismatches AS (
  -- Find bronze customer IDs not in silver
  SELECT COUNT(*) AS missing_in_target
  FROM retail_catalog.bronze.customers_raw b
  LEFT JOIN retail_catalog.silver.dim_customer s
    ON b.CustomerID = s.CustomerID AND s.IsActive = 1
  WHERE s.CustomerID IS NULL
),
orphans AS (
  -- Find silver customer IDs not in bronze
  SELECT COUNT(*) AS missing_in_source
  FROM retail_catalog.silver.dim_customer s
  LEFT JOIN retail_catalog.bronze.customers_raw b
    ON s.CustomerID = b.CustomerID
  WHERE s.IsActive = 1 AND b.CustomerID IS NULL
)
SELECT 
  m.missing_in_target,
  o.missing_in_source,
  CASE 
    WHEN m.missing_in_target = 0 AND o.missing_in_source = 0 THEN 'PASS: All customer IDs match'
    ELSE assert_true(FALSE, CONCAT('FAIL: ', CAST(m.missing_in_target AS STRING), ' customer IDs missing in target, ', CAST(o.missing_in_source AS STRING), ' customer IDs missing in source'))
  END AS validation_status
FROM mismatches m, orphans o;

In [0]:
%sql

--source to target validation productid mismatch
WITH mismatches AS (
  -- Find bronze product IDs not in silver
  SELECT COUNT(*) AS missing_in_target
  FROM retail_catalog.bronze.products_raw b
  LEFT JOIN retail_catalog.silver.dim_product s
    ON b.ProductID = s.ProductID
  WHERE s.ProductID IS NULL
),
orphans AS (
  -- Find silver product IDs not in bronze
  SELECT COUNT(*) AS missing_in_source
  FROM retail_catalog.silver.dim_product s
  LEFT JOIN retail_catalog.bronze.products_raw b
    ON s.ProductID = b.ProductID
  WHERE b.ProductID IS NULL
)
SELECT 
  m.missing_in_target,
  o.missing_in_source,
  CASE 
    WHEN m.missing_in_target = 0 AND o.missing_in_source = 0 THEN 'PASS: All product IDs match'
    ELSE assert_true(FALSE, CONCAT('FAIL: ', CAST(m.missing_in_target AS STRING), ' product IDs missing in target, ', CAST(o.missing_in_source AS STRING), ' product IDs missing in source'))
  END AS validation_status
FROM mismatches m, orphans o;